In [26]:
!pip install xgboost

In [68]:
!pip install optuna

  Using cached optuna-4.8.0-py3-none-any.whl.metadata (17 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
Using cached optuna-4.8.0-py3-none-any.whl (419 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.1 MB 3.7 MB/s eta 0:00:01
   ---------------------------------- ----- 1.8/2.1 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------  2.1/2.1 MB 4.9 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 2.9 MB/s  0:00:00
Using cached colorlog-6.10.1-py3-none-any.whl (11 kB)

   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ------ --------------------------------- 1/6 [greenlet]
   ----

In [77]:
!pip install lightgbm

In [108]:
!pip install catboost

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 1.0/100.2 MB 3.1 MB/s eta 0:00:32
    --------------------------------------- 1.8/100.2 MB 4.2 MB/s eta 0:00:24
   - -------------------------------------- 2.6/100.2 MB 3.6 MB/s eta 0:00:28
   - -------------------------------------- 3.1/100.2 MB 3.1 MB/s eta 0:00:32
   - -------------------------------------- 3.9/100.2 MB 3.2 MB/s eta 0:00:30
   - -------------------------------------- 4.2/100.2 MB 3.4 MB/s eta 0:00:29
   - -------------------------------------- 4.5/100.2 MB 2.7 MB/s eta 0:00:35
   -- ------------------------------------- 5.2/100.2 MB 2.8 MB/s eta 0:00:34
   -- ------------------------------------- 6.3/100.2 MB 3.0 MB/s eta 0:00:31
   -- ------------------------------------- 6.3/100.2 MB 3.0 MB/s eta 0:00:31
   -- -

In [80]:
import numpy as np
import pandas as pd

In [81]:
df=pd.read_csv('../Dataset/Developer_data')
df.sample()

,Age,EdLevel,Employment,WorkExp,LearnCodeChoose,LearnCodeAI,YearsCode,DevType,OrgSize,ICorPM,RemoteWork,PurchaseInfluence,AIThreat,NewRole,ToolCountWork,ToolCountPersonal,Country,AIModelsChoice,ConvertedCompYearly,JobSat
3629,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,5.0,"Yes, I am not new to coding but am learning ne...","Yes, I learned how to use AI-enabled tools for...",6.0,"Developer, desktop or enterprise applications","10,000 or more employees",Individual contributor,"Hybrid (some in-person, leans heavy to flexibi...",No,I'm not sure,I have somewhat considered changing my career ...,NaN,NaN,Ireland,Yes,61488.0,5.0


In [82]:
df.shape

(17892, 20)

In [83]:
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator,TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [84]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer,SimpleImputer

In [85]:
from sklearn.preprocessing import RobustScaler,PowerTransformer,OneHotEncoder
from sklearn.compose import make_column_selector

In [86]:
from sklearn.linear_model import BayesianRidge

In [87]:
from sklearn.model_selection import KFold,cross_val_score,cross_validate

In [88]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

In [89]:
from sklearn.metrics import r2_score,mean_absolute_error,mean_absolute_percentage_error,mean_squared_error,root_mean_squared_error

In [90]:
import optuna

In [91]:
from optuna.pruners import MedianPruner

In [92]:
from lightgbm import LGBMClassifier

In [93]:
X=df.drop(columns=['ConvertedCompYearly','JobSat'])
y=df['ConvertedCompYearly']

In [94]:
y_satisfaction = df["JobSat"]

In [95]:
def simplify_sat(val):
    if val <= 4: return "Low"
    if val <= 7: return "Medium"
    return "High"

y_sat_grouped = y_satisfaction.apply(simplify_sat)

In [96]:
from sklearn.model_selection import train_test_split
(X_train,X_test,y_train,y_test,y_sat_grouped_train,y_sat_grouped_test) = train_test_split(X,y,y_sat_grouped,test_size=0.2,random_state=42)

In [97]:
class change_data_type(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None):
        return self

    def transform(self,X):
        X=X.copy()
        for col in X.columns:
            if X[col].dtype in ['object']:
                X[col]=X[col].astype('category')
            else:
                X[col]=X[col]
        return X
        

In [98]:
num_pipeline = Pipeline([
    ('imputer', IterativeImputer(estimator=BayesianRidge(),max_iter=10,random_state=42)),
    ('power', PowerTransformer(method='yeo-johnson')),
    ('scaler', RobustScaler(quantile_range=(25, 75)))
])

In [99]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [100]:
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, make_column_selector(dtype_include='number')),
    ('cat', cat_pipeline, make_column_selector(dtype_include='category'))
],remainder='drop')

In [101]:
cat_pipeline2 = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

In [102]:
preprocessor2 = ColumnTransformer(transformers=[
    ('num', num_pipeline, make_column_selector(dtype_include='number')),
    ('cat', cat_pipeline2, make_column_selector(dtype_include='category'))
],remainder='drop')

In [103]:
num_pipeline3 = Pipeline([
    ('imputer', IterativeImputer(estimator=BayesianRidge(),max_iter=10,random_state=42)),
    ('power', PowerTransformer(method='yeo-johnson')),
    ('scaler', RobustScaler(quantile_range=(25, 75)))
])

In [104]:
cat_pipeline3 = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [105]:
preprocessor3 = ColumnTransformer(transformers=[
    ('num', num_pipeline3, make_column_selector(dtype_include='number')),
    ('cat', cat_pipeline3, make_column_selector(dtype_include='category'))
],remainder='drop')

.

.

# XGBoost:

In [106]:
from xgboost import XGBRegressor

model=XGBRegressor(
    n_estimators=500,
    learning_rate=0.01,
    max_depth=15,
    min_child_weight=4,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=3,
    objective="reg:squarederror",
    eval_metric='rmse',
    n_jobs=-1
    
)

In [107]:
xgb_pipeline = Pipeline([
    ('type_converter',change_data_type()),
    ('preprocessor', preprocessor),
    ('model', model)
])

In [27]:
CV=KFold(n_splits=5,shuffle=True,random_state=42)

In [28]:
cv_score=cross_val_score(xgb_pipeline,X,y,cv=CV,scoring='r2')

In [29]:
cv_score

array([0.58967384, 0.58115639, 0.57150163, 0.56481985, 0.57234587])

In [30]:
print("Std:", cv_score.std())

Std: 0.008626650881103851


In [31]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 370, 640),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.04),
        "max_depth": trial.suggest_int("max_depth", 12, 20),
        "min_child_weight": trial.suggest_int("min_child_weight", 3, 5),
        "subsample": trial.suggest_float("subsample", 0.7, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 0.9),
        "gamma": trial.suggest_int("gamma", 3, 5),
    }
    

    xgb_pipeline.set_params(
        model__n_estimators=params["n_estimators"],
        model__learning_rate=params["learning_rate"],
        model__max_depth=params["max_depth"],
        model__min_child_weight=params["min_child_weight"],
        model__subsample=params["subsample"],
        model__colsample_bytree=params["colsample_bytree"],
        model__gamma=params["gamma"]
    )
    
    scores = cross_val_score(xgb_pipeline, X, y, cv=CV, scoring='r2', n_jobs=-1)
    return scores.mean()

In [32]:
study = optuna.create_study(direction="maximize",pruner=MedianPruner())
study.optimize(objective, n_trials=30)

[I 2026-05-10 07:03:53,743] A new study created in memory with name: no-name-97221aaf-02d4-4805-baf2-d7d021d4aaad
[I 2026-05-10 07:04:05,264] Trial 0 finished with value: 0.5727086514843224 and parameters: {'n_estimators': 512, 'learning_rate': 0.0206549234438934, 'max_depth': 13, 'min_child_weight': 4, 'subsample': 0.8936409643073473, 'colsample_bytree': 0.8546239895690535, 'gamma': 3}. Best is trial 0 with value: 0.5727086514843224.
[I 2026-05-10 07:04:30,334] Trial 1 finished with value: 0.5571439662203894 and parameters: {'n_estimators': 554, 'learning_rate': 0.03476839849243881, 'max_depth': 17, 'min_child_weight': 3, 'subsample': 0.7723974889327632, 'colsample_bytree': 0.8784117486681544, 'gamma': 4}. Best is trial 0 with value: 0.5727086514843224.
[I 2026-05-10 07:04:43,867] Trial 2 finished with value: 0.5713134872816925 and parameters: {'n_estimators': 437, 'learning_rate': 0.022450512583846643, 'max_depth': 17, 'min_child_weight': 4, 'subsample': 0.738700200643295, 'colsample

In [108]:
print("Best Hyperparameters:", study.best_params)
print("Best CV Score:", study.best_value)

Best Hyperparameters: {'n_estimators': 577, 'learning_rate': 0.01442861298476298, 'max_depth': 12, 'min_child_weight': 4, 'subsample': 0.7030984480536636, 'colsample_bytree': 0.7919533894905313, 'gamma': 5}
Best CV Score: 0.5826038435542371


.

.

# LightGBM

In [109]:
from lightgbm import LGBMRegressor

model2 = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.01,
    max_depth=10,
    min_child_samples=10,
    subsample=0.8,
    colsample_bytree=0.8,
    min_split_gain=3,
    objective="regression",
    metric="rmse",
    n_jobs=-1
)

In [110]:
lgbm_pipeline = Pipeline([
    ('type_converter',change_data_type()),
    ('preprocessor', preprocessor),
    ('model2', model2)
])

In [92]:
model2_cv_score=cross_val_score(lgbm_pipeline,X,y,cv=CV,scoring='r2')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001780 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1065
[LightGBM] [Info] Number of data points in the train set: 14901, number of used features: 178
[LightGBM] [Info] Start training from score 82528.862694
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001897 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1003
[LightGBM] [Info] Number of data points in the train set: 14901, number of used features: 177
[LightGBM] [Info] Start training from score 82623.234548
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001832 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is

In [93]:
model2_cv_score

array([0.55341717, 0.57935313, 0.57366205, 0.58044032, 0.57108759])

In [94]:
print("Std:", model2_cv_score.std())

Std: 0.009730281459058937


In [37]:
def lgbm_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 370, 640),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.04),
        "max_depth": trial.suggest_int("max_depth", 8, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 20),
        "subsample": trial.suggest_float("subsample", 0.7, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 0.9),
        "min_split_gain": trial.suggest_float("min_split_gain", 1.0, 6.0),
    }
    

    lgbm_pipeline.set_params(
        model2__n_estimators=params["n_estimators"],
        model2__learning_rate=params["learning_rate"],
        model2__max_depth=params["max_depth"],
        model2__min_child_samples=params["min_child_samples"],
        model2__subsample=params["subsample"],
        model2__colsample_bytree=params["colsample_bytree"],
        model2__min_split_gain=params["min_split_gain"]
    )
    
    scores2 = cross_val_score(lgbm_pipeline, X, y, cv=CV, scoring='r2', n_jobs=-1)
    return scores2.mean()

In [38]:
study_lgbm = optuna.create_study(direction="maximize",pruner=MedianPruner())
study_lgbm.optimize(lgbm_objective, n_trials=40)

[I 2026-05-10 07:11:24,901] A new study created in memory with name: no-name-240cfc3d-7a60-413f-9e64-4ebcb6d71be5
[I 2026-05-10 07:11:28,799] Trial 0 finished with value: 0.5914512342469056 and parameters: {'n_estimators': 446, 'learning_rate': 0.03319501799245846, 'max_depth': 11, 'min_child_samples': 5, 'subsample': 0.8710220360098826, 'colsample_bytree': 0.815716504401275, 'min_split_gain': 2.389153519259665}. Best is trial 0 with value: 0.5914512342469056.
[I 2026-05-10 07:11:33,493] Trial 1 finished with value: 0.5881284522321666 and parameters: {'n_estimators': 601, 'learning_rate': 0.014092276789040376, 'max_depth': 8, 'min_child_samples': 16, 'subsample': 0.7227515894750117, 'colsample_bytree': 0.7662730875624818, 'min_split_gain': 2.0787407169610894}. Best is trial 0 with value: 0.5914512342469056.
[I 2026-05-10 07:11:37,749] Trial 2 finished with value: 0.5907357475653197 and parameters: {'n_estimators': 537, 'learning_rate': 0.013626363135653132, 'max_depth': 13, 'min_child_

In [111]:
print("Best Hyperparameters:", study_lgbm.best_params)
print("Best CV Score:", study_lgbm.best_value) 

Best Hyperparameters: {'n_estimators': 426, 'learning_rate': 0.03530504243591968, 'max_depth': 12, 'min_child_samples': 14, 'subsample': 0.7673977402284966, 'colsample_bytree': 0.7264545424876925, 'min_split_gain': 5.282313543165672}
Best CV Score: 0.5937550336322931


.

.

# GradientBoostingRegressor

In [112]:
from sklearn.ensemble import GradientBoostingRegressor

model3 = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.01,
    max_depth=10,
    min_samples_leaf=8,
    subsample=0.8,
    random_state=42
)

In [113]:
gbr_pipeline = Pipeline([
    ('type_converter',change_data_type()),
    ('preprocessor', preprocessor),
    ('model3', model3)
])

In [111]:
model3_cv_score=cross_val_score(gbr_pipeline,X,y,cv=CV,scoring='r2',n_jobs=-1)

In [112]:
model3_cv_score

array([0.55458362, 0.58076207, 0.5786639 , 0.5793177 , 0.57203018])

In [113]:
print("Std:", model3_cv_score.std())

Std: 0.009719267463775048


In [42]:
def gbr_objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 370, 640),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.04
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            8,
            16
        ),

        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf",
            5,
            20
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.7,
            0.9
        )
    }

    gbr_pipeline.set_params(
        model3__n_estimators=params["n_estimators"],
        model3__learning_rate=params["learning_rate"],
        model3__max_depth=params["max_depth"],
        model3__min_samples_leaf=params["min_samples_leaf"],
        model3__subsample=params["subsample"]
    )

    scores3 = cross_val_score(
        gbr_pipeline,
        X,
        y,
        cv=CV,
        scoring='r2',
        n_jobs=-1
    )

    return scores3.mean()

In [43]:
study_gbr = optuna.create_study(direction="maximize",pruner=MedianPruner())
study_gbr.optimize(gbr_objective,n_trials=10)

[I 2026-05-10 07:14:34,886] A new study created in memory with name: no-name-3bad64e9-2e83-4aec-bc22-5b4a57672c33
[I 2026-05-10 07:16:41,665] Trial 0 finished with value: 0.586928699240587 and parameters: {'n_estimators': 531, 'learning_rate': 0.01502798746329269, 'max_depth': 12, 'min_samples_leaf': 19, 'subsample': 0.8295230778379372}. Best is trial 0 with value: 0.586928699240587.
[I 2026-05-10 07:18:33,803] Trial 1 finished with value: 0.5836794357078128 and parameters: {'n_estimators': 471, 'learning_rate': 0.014082450507098717, 'max_depth': 11, 'min_samples_leaf': 13, 'subsample': 0.8675111439033008}. Best is trial 0 with value: 0.586928699240587.
[I 2026-05-10 07:19:47,163] Trial 2 finished with value: 0.5846809950897217 and parameters: {'n_estimators': 465, 'learning_rate': 0.03625060417795785, 'max_depth': 9, 'min_samples_leaf': 15, 'subsample': 0.879849257369899}. Best is trial 0 with value: 0.586928699240587.
[I 2026-05-10 07:21:25,595] Trial 3 finished with value: 0.5871629

In [44]:
print("Best Parameters:", study_gbr.best_params)
print("Best Score:", study_gbr.best_value)

Best Parameters: {'n_estimators': 621, 'learning_rate': 0.024894272925948965, 'max_depth': 9, 'min_samples_leaf': 17, 'subsample': 0.7984299573935234}
Best Score: 0.5871629399747741


.

.

# Random Forest Regressor

In [114]:
from sklearn.ensemble import RandomForestRegressor

model4 = RandomForestRegressor(
    n_estimators=400,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features='sqrt',
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

In [115]:
rf_pipeline = Pipeline([
    ('type_converter',change_data_type()),
    ('preprocessor', preprocessor),
    ('model4', model4)
])

In [125]:
model4_cv_score=cross_val_score(rfr_pipeline,X,y,cv=CV,scoring='r2',n_jobs=-1)

In [126]:
model4_cv_score

array([0.46686906, 0.47712866, 0.47501471, 0.49076269, 0.47081387])

In [127]:
print("Std:", model4_cv_score.std())

Std: 0.008128743608393479


In [47]:
def rf_objective(trial):

    params = {

        "n_estimators": trial.suggest_int(
            "n_estimators",
            400,
            600
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            7,
            20
        ),

        "min_samples_split": trial.suggest_int(
            "min_samples_split",
            2,
            10
        ),

        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf",
            1,
            10
        ),

        "max_features": trial.suggest_categorical(
            "max_features",
            ["sqrt", "log2"]
        ),

        "bootstrap": trial.suggest_categorical(
            "bootstrap",
            [True, False]
        )
    }

    rf_pipeline.set_params(
        model4__n_estimators=params["n_estimators"],
        model4__max_depth=params["max_depth"],
        model4__min_samples_split=params["min_samples_split"],
        model4__min_samples_leaf=params["min_samples_leaf"],
        model4__max_features=params["max_features"],
        model4__bootstrap=params["bootstrap"]
    )

    scores4 = cross_val_score(
        rf_pipeline,
        X,
        y,
        cv=CV,
        scoring='r2',
        n_jobs=-1
    )

    return scores4.mean()

In [48]:
study_rf = optuna.create_study(direction="maximize",pruner=MedianPruner())
study_rf.optimize(rf_objective,n_trials=10)

[I 2026-05-10 07:35:22,465] A new study created in memory with name: no-name-397351e8-2f1a-4e2f-ad73-dc0d3741062e
[I 2026-05-10 07:35:28,198] Trial 0 finished with value: 0.38180047836553577 and parameters: {'n_estimators': 506, 'max_depth': 8, 'min_samples_split': 3, 'min_samples_leaf': 5, 'max_features': 'log2', 'bootstrap': False}. Best is trial 0 with value: 0.38180047836553577.
[I 2026-05-10 07:36:13,321] Trial 1 finished with value: 0.5309400340153902 and parameters: {'n_estimators': 508, 'max_depth': 20, 'min_samples_split': 4, 'min_samples_leaf': 8, 'max_features': 'sqrt', 'bootstrap': False}. Best is trial 1 with value: 0.5309400340153902.
[I 2026-05-10 07:36:19,146] Trial 2 finished with value: 0.3544101459900245 and parameters: {'n_estimators': 514, 'max_depth': 7, 'min_samples_split': 9, 'min_samples_leaf': 10, 'max_features': 'log2', 'bootstrap': True}. Best is trial 1 with value: 0.5309400340153902.
[I 2026-05-10 07:36:38,207] Trial 3 finished with value: 0.47778832756465

In [49]:
print("Best Parameters:", study_rf.best_params)
print("Best Score:", study_rf.best_value)

Best Parameters: {'n_estimators': 411, 'max_depth': 19, 'min_samples_split': 9, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'bootstrap': False}
Best Score: 0.5414286882600295


.

.

# Cat gbm

In [354]:
from catboost import CatBoostRegressor

model5 = CatBoostRegressor(
    iterations=500,
    learning_rate=0.01,
    depth=8,
    loss_function='RMSE',
    verbose=0,
    random_state=42
)

In [355]:
cat_pipeline = Pipeline([
    ('type_converter',change_data_type()),
    ('preprocessor2', preprocessor),
    ('model5', model5)
])

In [133]:
model5_cv_score=cross_val_score(cat_pipeline,X,y,cv=CV,scoring='r2',n_jobs=-1)

In [134]:
model5_cv_score

array([0.52541731, 0.54368806, 0.53535626, 0.54948987, 0.53724648])

In [135]:
print("Std:", model5_cv_score.std())

Std: 0.008123612052721323


In [136]:
def cat_objective(trial):

    params = {

        "iterations": trial.suggest_int(
            "iterations",
            400,
            700
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.05
        ),

        "depth": trial.suggest_int(
            "depth",
            7,
            14
        ),

        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1,
            10
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        )
    }

    cat_pipeline.set_params(
        model5__iterations=params["iterations"],
        model5__learning_rate=params["learning_rate"],
        model5__depth=params["depth"],
        model5__l2_leaf_reg=params["l2_leaf_reg"],
        model5__subsample=params["subsample"]
    )

    scores5 = cross_val_score(
        cat_pipeline,
        X,
        y,
        cv=CV,
        scoring='r2',
        n_jobs=-1
    )

    return scores5.mean()

In [137]:
study_cat = optuna.create_study(direction="maximize",pruner=MedianPruner())
study_cat.optimize(cat_objective,n_trials=30)

[I 2026-05-09 18:24:00,137] A new study created in memory with name: no-name-ff28d2f9-4bba-4f6e-9617-f218a3f9d3d6
[I 2026-05-09 18:25:20,469] Trial 0 finished with value: 0.5532566035645252 and parameters: {'iterations': 592, 'learning_rate': 0.01162205408332135, 'depth': 11, 'l2_leaf_reg': 6.010780988021603, 'subsample': 0.8662773801356001}. Best is trial 0 with value: 0.5532566035645252.
[I 2026-05-09 18:25:43,774] Trial 1 finished with value: 0.5916176967232087 and parameters: {'iterations': 700, 'learning_rate': 0.04514005187647303, 'depth': 9, 'l2_leaf_reg': 7.550054043756495, 'subsample': 0.722806184146427}. Best is trial 1 with value: 0.5916176967232087.
[I 2026-05-09 18:26:00,401] Trial 2 finished with value: 0.5893250797592391 and parameters: {'iterations': 491, 'learning_rate': 0.047490129896185666, 'depth': 9, 'l2_leaf_reg': 3.7443462610939635, 'subsample': 0.882586405279645}. Best is trial 1 with value: 0.5916176967232087.
[I 2026-05-09 18:38:01,073] Trial 3 finished with v

In [138]:
print("Best Parameters:", study_cat.best_params)
print("Best Score:", study_cat.best_value)

Best Parameters: {'iterations': 666, 'learning_rate': 0.04972779117260329, 'depth': 9, 'l2_leaf_reg': 6.891692229561109, 'subsample': 0.6450592368815062}
Best Score: 0.592933187096734


.

.

# Stacking Pipeline

In [116]:
base_models = [
    ('lgbm', LGBMRegressor(**study_lgbm.best_params)),
    ('gbm', GradientBoostingRegressor(**study_gbr.best_params)), 
    ('xgb', XGBRegressor(**study.best_params))      
]

stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)

In [117]:
stacking_pipeline = Pipeline([
    ('type_converter', change_data_type()),
    ('preprocessor', preprocessor),
    ('stacker', stacking_model)
])

In [53]:
stack_cv_scores = cross_val_score(stacking_pipeline, X, y, cv=CV, scoring='r2', n_jobs=-1)

print(f"Stacking CV R2 Scores: {stack_cv_scores}")
print(f"Mean R2: {stack_cv_scores.mean():.4f}")

Stacking CV R2 Scores: [0.60914127 0.59383496 0.58857259 0.58476468 0.59408077]
Mean R2: 0.5941


.

## salary predictions for train 2

In [118]:
from sklearn.model_selection import cross_val_predict
salary_oof_train = cross_val_predict(stacking_pipeline,X_train,y_train,cv=5,n_jobs=-1)

.

.

## Train Model for predict Salary

In [119]:
stacking_pipeline.fit(X_train,y_train)

Pipeline(steps=[('type_converter', change_data_type()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   IterativeImputer(estimator=BayesianRidge(),
                                                                                    random_state=42)),
                                                                  ('power',
                                                                   PowerTransformer()),
                                                                  ('scaler',
                                                                   RobustScaler(quantile_range=(25,
                                                                                                75)))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000017...
                                                             importance_type=None,
                                                             interaction_constraints=None,
                                                             learning_rate=0.01442861298476298,
                                                             max_bin=None,
                                                             max_cat_threshold=None,
                                                             max_cat_to_onehot=None,
                                                             max_delta_step=None,
                                                             max_depth=12,
                                                             max_leaves=None,
                                                             min_child_weight=4,
                                                             missing=nan,
                                                             monotone_constraints=None,
                                                             multi_strategy=None,
                                                             n_estimators=577,
                                                             n_jobs=None,
                                                             num_parallel_tree=None, ...))],
                                   final_estimator=Ridge(), n_jobs=-1))])

.

.

## Predict Salary

In [120]:
y_pred_salary=stacking_pipeline.predict(X_test)

In [121]:
y_pred_salary

array([ 62606.16762143, 152307.60702545,  66963.43720022, ...,
       150830.31913701, 131019.94541709,  83198.19351119])

.

.

## Metrics

In [127]:
mae=mean_absolute_error(y_test,y_pred_salary)
mae

24559.90403141245

In [128]:
mse = mean_squared_error(y_test, y_pred_salary)
print(mse)

1139960414.2671905


In [129]:
rmse = np.sqrt(mean_squared_error(y_test,y_pred_salary))
print(rmse)

33763.299813069076


In [130]:
r2 = r2_score(y_test,y_pred_salary)
print(r2)

0.6092697032029829


In [131]:
mape = mean_absolute_percentage_error(y_test,y_pred_salary)
print(mape)

0.6409504511124989


.

.

## Data for JobSatisfaction

In [122]:
X_train_2 = X_train.copy()
X_train_2["pred_salary"] = salary_oof_train

X_test_2 = X_test.copy()
X_test_2["pred_salary"] = y_pred_salary

.

.

.

# Predict Job Satisfaction

In [123]:
sat_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,          
    n_estimators=500,
    learning_rate=0.01,
    max_depth=12,
    num_leaves=8,
    random_state=42
)

In [124]:
satisfaction_pipeline = Pipeline([
    ('type_converter', change_data_type()),
    ('preprocessor3', preprocessor3),
    ('classifier', sat_model)
])

In [125]:
satisfaction_pipeline.fit(X_train_2,y_sat_grouped_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000807 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1162
[LightGBM] [Info] Number of data points in the train set: 14313, number of used features: 125
[LightGBM] [Info] Start training from score -0.691263
[LightGBM] [Info] Start training from score -2.358844
[LightGBM] [Info] Start training from score -0.905036


Pipeline(steps=[('type_converter', change_data_type()),
                ('preprocessor3',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   IterativeImputer(estimator=BayesianRidge(),
                                                                                    random_state=42)),
                                                                  ('power',
                                                                   PowerTransformer()),
                                                                  ('scaler',
                                                                   RobustScaler(quantile_range=(25,
                                                                                                75)))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x000001...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000017CE7F37A90>)])),
                ('classifier',
                 LGBMClassifier(learning_rate=0.01, max_depth=12,
                                n_estimators=500, num_class=3, num_leaves=8,
                                objective='multiclass', random_state=42))])

In [64]:
satisfaction_pred = satisfaction_pipeline.predict(X_test_2)

.

## Metrics

In [132]:
from sklearn.metrics import accuracy_score
accuracy_score(y_sat_grouped_test,satisfaction_pred)

0.5741827326068735

In [133]:
from sklearn.metrics import classification_report
print(classification_report(y_sat_grouped_test,satisfaction_pred))

              precision    recall  f1-score   support

        High       0.62      0.74      0.67      1827
         Low       0.50      0.01      0.01       296
      Medium       0.50      0.48      0.49      1456

    accuracy                           0.57      3579
   macro avg       0.54      0.41      0.39      3579
weighted avg       0.56      0.57      0.55      3579



.

.

# Save Model

In [135]:
import joblib

In [136]:
models = {
    "salary_model": stacking_pipeline,
    "satisfaction_model": satisfaction_pipeline
}

joblib.dump(models, "full_system.pkl")

['full_system.pkl']

# load model 

In [137]:
models = joblib.load("full_system.pkl")

salary_model = models["salary_model"]
satisfaction_model = models["satisfaction_model"]

.

.

# Prediction 

In [138]:
X_new = pd.DataFrame([{
    "Age": "25-34 years old",
    "EdLevel": "Bachelor’s degree (B.A., B.S., B.Eng., etc.)",
    "Employment": "Employed",
    "WorkExp": 3.0,
    "LearnCodeChoose": "Yes, I am not new to coding but am learning new things",
    "LearnCodeAI": "Yes, I learned how to use AI-enabled tools for work",
    "YearsCode": 5.0,
    "DevType": "Developer, full-stack",
    "OrgSize": "10,000 or more employees",
    "ICorPM": "Individual contributor",
    "RemoteWork": "Hybrid (some in-person, leans heavy to flexibility)",
    "PurchaseInfluence": "No",
    "AIThreat": "No",
    "NewRole": "I have transitioned into a new career and/or industry",
    "ToolCountWork": 3.0,
    "ToolCountPersonal": 5.0,
    "Country": "United States of America",
    "AIModelsChoice": "No"
}])

In [141]:
class UnifiedDeveloperModel(BaseEstimator, ClassifierMixin):
    def __init__(self, salary_model, satisfaction_model):
        self.salary_model = salary_model
        self.satisfaction_model = satisfaction_model
        
    def predict(self, X_new):
        X_df = pd.DataFrame(X_new).copy()
        
        # 1. Get Salary Prediction
        salary_preds = self.salary_model.predict(X_df)
        
        # 2. Update data for Job satisfaction 
        X_df["pred_salary"] = salary_preds
        
        # 3. Get Satisfaction Prediction
        satisfaction_preds = self.satisfaction_model.predict(X_df)
        
        return satisfaction_preds, salary_preds


.

.

# --- Deployment ---

In [144]:
full_model = UnifiedDeveloperModel(
    salary_model=stacking_pipeline, 
    satisfaction_model=satisfaction_pipeline
)


In [145]:
sat_pred, sal_pred = full_model.predict(X_new)
print(f"Predicted Salary: {sal_pred[0]}")
print(f"Predicted Satisfaction: {sat_pred[0]}") 

Predicted Salary: 100611.62851660873
Predicted Satisfaction: High


.

#                                                             THE END

.